In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df=pd.read_excel("/content/dataset.xlsx")

In [ ]:


# 1. shape
print("Shape:", df.shape)

# 2. missing values
print("Missing values:\n", df.isnull().sum())

# 3. preview data
print(df.head())

Shape: (100, 93)
Missing values:
 s_no           0
age            0
gender         0
birth_order    0
religioun      0
              ..
PSY_8          0
PSY_9          0
PSY_10         0
PSY_11         0
PSY_12         0
Length: 93, dtype: int64
   s_no  age  gender  birth_order  religioun  category  family_type  \
0     1    1       2            1          1         1            1   
1     2    1       2            3          1         1            1   
2     3    1       2            1          1         1            2   
3     4    1       1            3          1         1            1   
4     5    1       1            1          1         1            1   

   study_year  residence  present_living  ...  PSY_3  PSY_4  PSY_5  PSY_6  \
0           1          1               1  ...      3      5      5      5   
1           2          2               1  ...      4      5      3      4   
2           2          1               1  ...      3      2      5      3   
3           2      

# Create EN_score

In [ ]:
# select EN columns
en_cols = [f"EN_{i}" for i in range(1, 23)]

# create EN_score
df["EN_score"] = df[en_cols].mean(axis=1)

# check result
print(df[["EN_score"]].head())

   EN_score
0  2.863636
1  3.318182
2  3.636364
3  3.772727
4  5.000000


In [ ]:
print(df["EN_score"].describe())

count    100.000000
mean       3.086818
std        0.499106
min        1.681818
25%        2.772727
50%        3.045455
75%        3.375000
max        5.000000
Name: EN_score, dtype: float64


# **Create Anxiety Score**
For Emotional Neglect:
👉 Only 1 value per question

But for Anxiety:
👉 Each question has 2 values

Anxiety (IA_AN) and
Avoidance (IA_AV)

🧠 What does this mean?

👉 A person may:

Feel anxiety ❗
Avoid situation ❗

Both together = real interpersonal anxiety

💡 So what should we do?

👉 Combine both:

Total Anxiety = Anxiety + Avoidance

Then average.

First variable:
IA_AN → 0 to 3
Second variable:
IA_AV → 0 to 3

➕ When we ADD:

👉 Maximum happens when both are maximum:

3 + 3 = 6 ✅

In [ ]:
# Anxiety columns
an_cols = [f"IA_AN_{i}" for i in range(1, 25)]
av_cols = [f"IA_AV_{i}" for i in range(1, 25)]

# total anxiety per question
df["Anxiety_score"] = (df[an_cols].values + df[av_cols].values).mean(axis=1)

# check
print(df[["Anxiety_score"]].head())

   Anxiety_score
0       2.250000
1       2.708333
2       2.041667
3       2.541667
4       4.208333


Create PsyCap Score (Resilience / Hope / Confidence)

🎯 WHY we are doing this?

👉 PsyCap (Psychological Capital) represents:

1.   Hope
2.   Confidence
3.  Resilience
4.  Optimism

👉 This is VERY IMPORTANT because:

High Anxiety ❌ alone is not enough

But Low PsyCap + High Anxiety = Dangerous combination

In [ ]:
# PsyCap columns
psy_cols = [f"PSY_{i}" for i in range(1, 13)]

# create PsyCap score
df["PsyCap_score"] = df[psy_cols].mean(axis=1)

# check
print(df[["PsyCap_score"]].head())

   PsyCap_score
0      4.250000
1      3.833333
2      3.500000
3      3.416667
4      1.583333


In [ ]:
# create new dataset with only 3 features
processed_df = df[["EN_score", "Anxiety_score", "PsyCap_score"]]

# save it
processed_df.to_csv("processed_data.csv", index=False)

# **Scaling**

Right now your features are:

EN_score → 1 to 5

Anxiety_score → 0 to 6

PsyCap_score → 1 to 5

👉 Problem:
These are in different ranges

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = df[["EN_score", "Anxiety_score", "PsyCap_score"]]
X_scaled = scaler.fit_transform(X)

# check
print(X_scaled[:5])

[[-0.4494161   0.07705418  1.60053431]
 [ 0.46589164  0.60026159  0.81518872]
 [ 1.10660707 -0.16076737  0.18691225]
 [ 1.38119939  0.41000435  0.02984313]
 [ 3.85253031  2.31257675 -3.42567746]]


Model may create groups like:

Group 0 → High anxiety, low PsyCap 🔴

Group 1 → Moderate 🟡

Group 2 → Healthy 🟢


📌 First Decision: How many groups?

👉 We choose:

K = 3

👉 Why?

High

Medium

Low

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)

# apply clustering
df["cluster"] = kmeans.fit_predict(X_scaled)

# check result
print(df[["EN_score", "Anxiety_score", "PsyCap_score", "cluster"]].head())

   EN_score  Anxiety_score  PsyCap_score  cluster
0  2.863636       2.250000      4.250000        1
1  3.318182       2.708333      3.833333        2
2  3.636364       2.041667      3.500000        2
3  3.772727       2.541667      3.416667        2
4  5.000000       4.208333      1.583333        2


In [ ]:
df.groupby("cluster")[["EN_score", "Anxiety_score", "PsyCap_score"]].mean()

,EN_score,Anxiety_score,PsyCap_score
cluster,,,
0,3.144269,1.614130,2.833333
1,2.770130,1.654762,3.833333
2,3.319264,2.933532,3.351190


# Cluster 2

High emotional neglect

High anxiety

Moderate resilience

👉 THIS IS HIGH RISK GROUP

# Cluster 1

Low neglect

Low anxiety

High resilience

👉 THIS IS LOW RISK / HEALTHY GROUP

# Cluster 0

Moderate neglect

Low anxiety

Low PsyCap

👉 ⚠️ Interesting group:

Not anxious yet

But low resilience → can become risky later

👉 MODERATE / VULNERABLE GROUP

In [ ]:
def label_cluster(cluster):
    if cluster == 2:
        return "High Risk"
    elif cluster == 0:
        return "Moderate / Vulnerable"
    else:
        return "Low Risk"

df["risk_level"] = df["cluster"].apply(label_cluster)

# **Therapy Recommendation Logic**



In [ ]:
def recommend_therapy(risk):
    if risk == "High Risk":
        return "Intensive Group Therapy (2 sessions/week)"
    elif risk == "Moderate / Vulnerable":
        return "Weekly Support Group"
    else:
        return "Well-being & Awareness Session"

df["therapy_plan"] = df["risk_level"].apply(recommend_therapy)

# check
print(df[["risk_level", "therapy_plan"]].head())

  risk_level                               therapy_plan
0   Low Risk             Well-being & Awareness Session
1  High Risk  Intensive Group Therapy (2 sessions/week)
2  High Risk  Intensive Group Therapy (2 sessions/week)
3  High Risk  Intensive Group Therapy (2 sessions/week)
4  High Risk  Intensive Group Therapy (2 sessions/week)


In [ ]:
df["group_id"] = df["cluster"].map({
    2: "Group A (High Risk)",
    0: "Group B (Moderate)",
    1: "Group C (Low Risk)"
})

# **Prediction Function **

In [ ]:
def predict_user(en_answers, ia_an_answers, ia_av_answers, psy_answers):

    import numpy as np

    # Step 1: Calculate scores
    EN_score = np.mean(en_answers)
    Anxiety_score = np.mean(np.array(ia_an_answers) + np.array(ia_av_answers))
    PsyCap_score = np.mean(psy_answers)

    # Step 2: Prepare input
    X_new = [[EN_score, Anxiety_score, PsyCap_score]]

    # Step 3: Scale
    X_new_scaled = scaler.transform(X_new)

    # Step 4: Predict cluster
    cluster = kmeans.predict(X_new_scaled)[0]

    # Step 5: Map risk
    if cluster == 2:
        risk = "High Risk"
    elif cluster == 0:
        risk = "Moderate / Vulnerable"
    else:
        risk = "Low Risk"

    # Step 6: Therapy
    if risk == "High Risk":
        therapy = "Intensive Group Therapy (2 sessions/week)"
        therapist = "Dr. A (Clinical Psychologist)"
    elif risk == "Moderate / Vulnerable":
        therapy = "Weekly Support Group"
        therapist = "Dr. B (Counselor)"
    else:
        therapy = "Well-being Session"
        therapist = "Dr. C (Wellness Coach)"

    # Step 7: Group
    group = f"Group {cluster}"

    return {
        "EN_score": EN_score,
        "Anxiety_score": Anxiety_score,
        "PsyCap_score": PsyCap_score,
        "risk": risk,
        "therapy": therapy,
        "therapist": therapist,
        "group": group
    }

In [ ]:
import pickle

# save kmeans model
with open("kmeans_model.pkl", "wb") as f:
    pickle.dump(kmeans, f)

# save scaler
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)